# Aiso Hotel — Prism end-to-end demo

Этот ноутбук показывает полный жизненный цикл `Prism` на крошечном русскоязычном корпусе про отель:

1. инициализация (SONAR embedder + Qdrant + OpenAI LLM);
2. ingest с автодетектом языка;
3. инспекция графа (узлы, ключевые фразы, corpus summary);
4. несколько поисковых запросов через `prism.search`;
5. end-to-end ответ через `prism.answer`;
6. демо отказа на запросе на «не-том» языке (англ. → ру).

**Что нужно перед запуском:**
- Qdrant поднят локально: `docker compose up qdrant`;
- `OPENAI_API_KEY` в `.env`;
- установлен опциональный `sonar`-extra (`pip install -e ".[sonar]"`).

## 1. Импорты и окружение

In [1]:
import logging
import os
from pathlib import Path

from dotenv import load_dotenv
from qdrant_client import AsyncQdrantClient

from prism import (
    LLMClient,
    MarkdownChunker,
    Prism,
    PrismGraph,
    QdrantBackend,
    SonarEmbedder,
)

load_dotenv()
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY must be set in .env"

logging.basicConfig(
    level="INFO",
    format="%(asctime)s %(levelname)s %(name)s: %(message)s",
)
logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("fairseq2").setLevel(logging.WARNING)

## 2. Подготовка корпуса

`demo/aiso_hotel.txt` — плоский текст с тремя смысловыми разделами. Чтобы `MarkdownChunker` мог разбить его на чанки, поднимаем заголовки разделов до markdown-уровня.

In [2]:
DATA_FILE = Path.cwd() / "demo" / "aiso_hotel.txt"
if not DATA_FILE.exists():
    DATA_FILE = Path.cwd() / "aiso_hotel.txt"

_SECTION_HEADERS = ("Описание отеля", "Услуги и удобства", "Правила проживания")

def load_corpus() -> str:
    raw = DATA_FILE.read_text(encoding="utf-8")
    lines: list[str] = []
    for line in raw.splitlines():
        if line.strip() in _SECTION_HEADERS:
            lines.append(f"## {line.strip()}")
        else:
            lines.append(line)
    return "# Aiso Hotel\n\n" + "\n".join(lines) + "\n"

corpus = load_corpus()
print(corpus[:600], "...")

# Aiso Hotel

Отель Aiso Hotel
Адрес: г. Москва, ул. Новочеркасская, д. 198, строение 1
Категория: 4 звезды
Время заезда: с 14:00
Время выезда: до 12:00
Телефон: +7 (495) 123-45-67
E-mail: info@aiso-hotel.ru
Сайт: www.aiso-hotel.ru

## Описание отеля
Отель «Aiso Hotel» — это уютное городское убежище с современным интерьером, высоким уровнем сервиса и тёплой атмосферой. Отель расположен в историческом районе Москвы с удобной транспортной доступностью: 10 минут до станции метро, 20 минут на автомобиле до центра города. В шаговой доступности — парки, архитектурные памятники и гастрономические ули ...


## 3. Инициализация компонентов

- `SonarEmbedder` — 1024-мерные эмбеддинги, язык будет проставлен автоматически в `Prism._set_language`.
- `QdrantBackend` — обёртка над `AsyncQdrantClient`; коллекция создаётся (`recreate=True`) с нуля, чтобы запуски были воспроизводимы.
- `LLMClient` — `temperature=None`, т.к. свежие модели GPT-5 не принимают этот параметр.
- `Prism` — оркестратор. Язык можно явно передать в `Prism(..., language="ru")`, но мы оставим автодетект.

In [3]:
embedder = SonarEmbedder(device=os.getenv("PRISM_SONAR_DEVICE", "cpu"))
qdrant = QdrantBackend(
    AsyncQdrantClient(url=os.getenv("QDRANT_URL", "http://localhost:6333")),
    collection_name="prism_aiso_hotel_demo",
)
graph = await PrismGraph.create(qdrant, embedder, recreate=True)
llm = LLMClient(temperature=None)
prism = Prism(
    graph,
    llm,
    chunker=MarkdownChunker(max_tokens=256, min_section_tokens=10),
)
print("Prism initialised; language =", prism.language)

/home/vadim/snap/miniconda3/envs/prism/lib/python3.12/site-packages/qdrant_client/async_qdrant_remote.py:225: UserWarning: Qdrant client version 1.18.0 is incompatible with server version 1.12.4. Major versions should match and minor version difference must not exceed 1. Set check_compatibility=False to skip version check.
  show_warning(
2026-06-11 13:42:33,266 INFO prism.storage.qdrant: created Qdrant collection prism_aiso_hotel_demo (dim=1024, distance=Cosine)


Prism initialised; language = None


## 4. Ingest

`prism.ingest(markdown)` выполняет:

1. автоопределение языка (на первом ingest, если не задан явно) → `Prism._set_language` синхронизирует язык эмбеддера, промптов и лексического пайплайна;
2. чанкование markdown по заголовкам;
3. LLM-извлечение `(header, summary, key_phrases)` для каждого чанка;
4. батч-эмбеддинг ключевых фраз, дедупликация, аплоад в Qdrant;
5. построение corpus summary (используется как DATA CONTEXT при будущих запросах).

In [4]:
nodes = await prism.ingest(corpus)
print(f"language: {prism.language}")
print(f"nodes:    {len(nodes)}")
for n in nodes:
    print(f"  - {n.name}: {n.keypoints[:5]}")
print()
print("corpus summary:")
print(prism.corpus_summary)

2026-06-11 13:42:38,861 INFO prism.core.engine: Prism: language=ru (embedder source_lang=rus_Cyrl)
2026-06-11 13:42:39,033 INFO prism.core.engine: ingest: extracting 4 chunks
2026-06-11 13:42:42,097 INFO prism.core.engine: ingest: indexing 4 nodes


Output()

2026-06-11 13:42:52,514 INFO prism.storage.qdrant: upserted 27 points to prism_aiso_hotel_demo in 1 batches
2026-06-11 13:42:55,832 INFO prism.core.engine: ingest: corpus summary refreshed (512 chars)


language: ru
nodes:    4
  - Aiso Hotel: ['Aiso Hotel']
  - Описание отеля: ['Описание отеля', 'Aiso Hotel']
  - Услуги и удобства: ['Услуги и удобства', 'Aiso Hotel', 'бесплатный Wi-Fi', 'круглосуточная регистрация', 'трансфер аэропорта']
  - Правила проживания: ['Правила проживания', 'Aiso Hotel', 'режим тишины', 'домашние животные', 'запрет курения']

corpus summary:
Корпус посвящён отелю Aiso Hotel и содержит краткую справочную информацию для гостей. Основные темы: общее описание отеля, доступные услуги и удобства (например, бесплатный Wi‑Fi, круглосуточная регистрация, трансфер из аэропорта) и правила проживания. По структуре это небольшая карточка/мини-справочник об отеле с разделами «Описание», «Услуги и удобства» и «Правила проживания». Из корпуса, вероятно, можно отвечать на вопросы о сервисах отеля и условиях размещения, но не о более широких туристических темах.


## 5. Поиск

`prism.search(query)` под капотом:

- LLM раскладывает запрос на `key_phrases` + `synonyms`;
- по каждой фразе идёт vector search в Qdrant + 1-hop расширение по соседям;
- параллельно — strict-AND лексический поиск по стемам с sentence-level proximity;
- результаты дедуплицируются, реконструируются в параграфы;
- LLM-судья отсеивает нерелевантные фрагменты (`filter_relevance=True`).

Покажем несколько типов запросов — точечный («можно ли с собакой»), про время («во сколько заезд»), широкий («какие удобства»), про несуществующее («тренажёрный зал»).

In [5]:
QUERIES = [
    "можно ли с собакой?",
    "во сколько заезд и выезд?",
    "какие удобства есть в отеле?",
    "есть ли тренажёрный зал?",
]

for q in QUERIES:
    print("=" * 60)
    print(f"QUERY: {q}")
    res = await prism.search(q)
    print(f"keypoints: {res.keypoints}")
    if res.note:
        print(f"note:      {res.note}")
    for i, p in enumerate(res.paragraphs, 1):
        print(f"  [{i}] {p}")
    print()

2026-06-11 13:42:55,862 INFO prism.core.engine: search: query='можно ли с собакой?'


QUERY: можно ли с собакой?


2026-06-11 13:42:57,005 INFO prism.core.engine: search: keypoints=['размещение с собакой', 'с питомцами', 'проживание с собакой']
2026-06-11 13:42:57,216 INFO prism.core.graph: vector search: 2 nodes: [(3, 'Услуги и удобства'), (4, 'Правила проживания')]
2026-06-11 13:42:57,232 INFO prism.core.graph: lexical search: 0 nodes: []
2026-06-11 13:42:57,234 INFO prism.core.engine: search: hybrid retrieval -> 2 unique nodes (vector=2, +neighbors=0, lexical=0)
2026-06-11 13:42:57,235 INFO prism.core.engine: search: reconstructed 2 paragraphs from 2 nodes
2026-06-11 13:42:58,302 INFO prism.core.engine: search: relevance filter kept 1/2 paragraphs
2026-06-11 13:42:58,304 INFO prism.core.engine: search: query='во сколько заезд и выезд?'


keypoints: ['размещение с собакой', 'с питомцами', 'проживание с собакой']
  [1] - Курение на территории отеля строго запрещено - Проживание с домашними животными допускается. Животное должно быть до 5 кг.

QUERY: во сколько заезд и выезд?


2026-06-11 13:42:59,359 INFO prism.core.engine: search: keypoints=['время заезда', 'время выезда', 'заезд и выезд', 'часы заезда', 'часы выезда', 'время регистрации']
2026-06-11 13:42:59,763 INFO prism.core.graph: vector search: 2 nodes: [(3, 'Услуги и удобства'), (4, 'Правила проживания')]
2026-06-11 13:42:59,785 INFO prism.core.graph: lexical search: 1 nodes: [(1, 'Aiso Hotel')]
2026-06-11 13:42:59,786 INFO prism.core.engine: search: hybrid retrieval -> 3 unique nodes (vector=2, +neighbors=0, lexical=1)
2026-06-11 13:42:59,786 INFO prism.core.engine: search: reconstructed 3 paragraphs from 3 nodes
2026-06-11 13:43:00,895 INFO prism.core.engine: search: relevance filter kept 1/3 paragraphs
2026-06-11 13:43:00,896 INFO prism.core.engine: search: query='какие удобства есть в отеле?'


keypoints: ['время заезда', 'время выезда', 'заезд и выезд', 'часы заезда', 'часы выезда', 'время регистрации']
  [1] Время заезда: с 14:00 Время выезда: до 12:00

QUERY: какие удобства есть в отеле?


2026-06-11 13:43:01,918 INFO prism.core.engine: search: keypoints=['удобства отеля', 'услуги отеля', 'сервисы отеля']
2026-06-11 13:43:02,285 INFO prism.core.graph: vector search: 3 nodes: [(2, 'Описание отеля'), (3, 'Услуги и удобства'), (4, 'Правила проживания')]
2026-06-11 13:43:02,298 INFO prism.core.graph: lexical search: 1 nodes: [(2, 'Описание отеля')]
2026-06-11 13:43:02,299 INFO prism.core.engine: search: hybrid retrieval -> 3 unique nodes (vector=3, +neighbors=0, lexical=1)
2026-06-11 13:43:02,300 INFO prism.core.engine: search: reconstructed 3 paragraphs from 3 nodes
2026-06-11 13:43:03,864 INFO prism.core.engine: search: relevance filter kept 1/3 paragraphs
2026-06-11 13:43:03,865 INFO prism.core.engine: search: query='есть ли тренажёрный зал?'


keypoints: ['удобства отеля', 'услуги отеля', 'сервисы отеля']
  [1] - Бесплатный Wi-Fi на всей территории - Круглосуточная стойка регистрации - Кафе с завтраками по будням и бранчами по выходным - Трансфер от/до аэропорта (по запросу) - Консьерж-сервис - Прачечная и химчистка - Просторная лаунж-зона с библиотекой и рабочими столами - Мини-фитнес-зал - Зона для coworking - Камера хранения багажа

QUERY: есть ли тренажёрный зал?


2026-06-11 13:43:05,195 INFO prism.core.engine: search: keypoints=['тренажёрный зал', 'спортзал', 'тренажёрка']
2026-06-11 13:43:05,497 INFO prism.core.graph: vector search: 2 nodes: [(3, 'Услуги и удобства'), (4, 'Правила проживания')]
2026-06-11 13:43:05,508 INFO prism.core.graph: lexical search: 0 nodes: []
2026-06-11 13:43:05,509 INFO prism.core.engine: search: hybrid retrieval -> 2 unique nodes (vector=2, +neighbors=0, lexical=0)
2026-06-11 13:43:05,510 INFO prism.core.engine: search: reconstructed 2 paragraphs from 2 nodes
2026-06-11 13:43:06,312 INFO prism.core.engine: search: relevance filter kept 1/2 paragraphs


keypoints: ['тренажёрный зал', 'спортзал', 'тренажёрка']
  [1] - Мини-фитнес-зал



## 6. End-to-end ответ

`prism.answer(query)` = `search()` + LLM-суммаризация по найденным фрагментам в одном вызове. Возвращает `AnswerResult` с готовым ответом + `final_summary` (каталожная сводка) + ссылкой на исходный `SearchResult` для цитирования.

In [6]:
ans = await prism.answer("можно ли с собакой?")
print("answer:       ", ans.answer)
print("final_summary:", ans.final_summary)
if ans.note:
    print("note:         ", ans.note)
print()
print("source paragraphs:")
for p in (ans.search.paragraphs if ans.search else []):
    print(f"  - {p}")

2026-06-11 13:43:09,988 INFO prism.core.engine: search: query='можно ли с собакой?'
2026-06-11 13:43:11,136 INFO prism.core.engine: search: keypoints=['размещение с собакой', 'собака в отеле', 'питомцы в отеле']
2026-06-11 13:43:11,368 INFO prism.core.graph: vector search: 3 nodes: [(2, 'Описание отеля'), (3, 'Услуги и удобства'), (4, 'Правила проживания')]
2026-06-11 13:43:11,382 INFO prism.core.graph: lexical search: 0 nodes: []
2026-06-11 13:43:11,383 INFO prism.core.engine: search: hybrid retrieval -> 3 unique nodes (vector=3, +neighbors=0, lexical=0)
2026-06-11 13:43:11,383 INFO prism.core.engine: search: reconstructed 3 paragraphs from 3 nodes
2026-06-11 13:43:12,277 INFO prism.core.engine: search: relevance filter kept 1/3 paragraphs


answer:        Да, проживание с собакой допускается, если вес животного не превышает 5 кг.
final_summary: Данные об условиях проживания с домашними животными

source paragraphs:
  - - Курение на территории отеля строго запрещено - Проживание с домашними животными допускается. Животное должно быть до 5 кг.


## 7. Кросс-язычный запрос

Если запрос приходит на языке, отличном от корпусного, `Prism` сразу возвращает локализованное сообщение на языке запроса — без vector/lexical поиска и без LLM-вызовов. Это политика «вежливого отказа» вместо попытки гонять SONAR через языковой шов.

In [ ]:
en = await prism.answer("can I bring my dog?", query_language="en")
print("answer:", en.answer)
print("note:  ", en.note)

answer: The corpus is available in Русский only. Please ask your question in this language.
note:   language mismatch


## 8. Cleanup

In [8]:
await qdrant.client.close()
print("qdrant client closed")

qdrant client closed
